In [1]:
import numpy as np
bohr_to_ang = 0.529177249 # Angstrom / Bohr

# Equilibrium Atomic Positions from Quantum ESPRESSO nscf Output

Below are the equilibrium atomic positions and lattice vectors for MnBi2Te4 in Angstroms. They are from the cif file. The lattice vectors are for the *rhombohedral unit cell*.

In [2]:
a0 = 27.39848
alpha = np.deg2rad(9.07191)

# high-precision components
z = a0 * np.sqrt((1 + 2*np.cos(alpha)) / 3)
rho = a0 * np.sqrt((2/3) * (1 - np.cos(alpha)))

# rotated
a1 = np.array([ np.sqrt(3)/2 * rho, -0.5 * rho, z])
a2 = np.array([ 0.0,                rho,        z])
a3 = np.array([-np.sqrt(3)/2 * rho, -0.5 * rho, z])

A = np.vstack([a1, a2, a3])
A

array([[ 2.16679999, -1.25100256, 27.28400038],
       [ 0.        ,  2.50200512, 27.28400038],
       [-2.16679999, -1.25100256, 27.28400038]])

In [3]:
positions_alat = [
    ("Mn1", [0.0000000, 0.0000000, 0.0000000]),
    ("Mn2", [0.0000000, 0.0000000, 1.4937325]),
    ("Bi", [0.0000000, 0.0000000, 0.6346869]),
    ("Bi", [0.0000000, 0.0000000, 2.1284194]),
    ("Bi", [0.0000000, 0.0000000, 0.8590455]),
    ("Bi", [0.0000000, 0.0000000, 2.3527780]),
    ("Te", [0.0000000, 0.0000000, 0.1992042]),
    ("Te", [0.0000000, 0.0000000, 1.6929366]),
    ("Te", [0.0000000, 0.0000000, 1.2945283]),
    ("Te", [0.0000000, 0.0000000, 2.7882608]),
    ("Te", [0.0000000, 0.0000000, 0.4392470]),
    ("Te", [0.0000000, 0.0000000, 1.9329794]),
    ("Te", [0.0000000, 0.0000000, 1.0544855]),
    ("Te", [0.0000000, 0.0000000, 2.5482179]),
] # units of alat

In [4]:
positions_ang = [
    (atom, (np.array(coords) * a0).tolist())
    for atom, coords in positions_alat
]

In [5]:
positions_ang

[('Mn1', [0.0, 0.0, 0.0]),
 ('Mn2', [0.0, 0.0, 40.92600002659999]),
 ('Bi', [0.0, 0.0, 17.389456335912]),
 ('Bi', [0.0, 0.0, 58.315456362511995]),
 ('Bi', [0.0, 0.0, 23.53654095084]),
 ('Bi', [0.0, 0.0, 64.46254097744]),
 ('Te', [0.0, 0.0, 5.457892289616]),
 ('Te', [0.0, 0.0, 46.383889576368]),
 ('Te', [0.0, 0.0, 35.468107736984]),
 ('Te', [0.0, 0.0, 76.394107763584]),
 ('Te', [0.0, 0.0, 12.03470014456]),
 ('Te', [0.0, 0.0, 52.960697431312]),
 ('Te', [0.0, 0.0, 28.89129988204]),
 ('Te', [0.0, 0.0, 69.817297168792])]

In [6]:
import re
def auto_id_positions(items):
    """
    items: [(species, [x,y,z]), ...]
    returns: [(id, species, [x,y,z])]
    
    - Keeps species exactly as given (e.g., "Mn1" stays "Mn1")
    - Adds numeric suffix only if species doesn't already end with digits
    """
    counts = {}
    out = []
    for sp, r in items:
        # Check if species already ends with digits
        if re.search(r"\d+$", sp):
            aid = sp
        else:
            counts[sp] = counts.get(sp, 0) + 1
            aid = f"{sp}{counts[sp]}"
        out.append((aid, sp, list(map(float, r))))
    return out

def make_qe_pos_block(positions, displacements=None, amp=0.0, unit="angstrom"):
    """
    positions: [(id, species, [x,y,z]), ...]
    displacements: {id: [dx,dy,dz]}  (optional)
                   or a callable  disp(id, species, pos) -> [dx,dy,dz]
    amp: scale factor for displacement
    """
    lines = [f"ATOMIC_POSITIONS {unit}"]
    use_callable = callable(displacements)
    for aid, sp, r in positions:
        r = np.array(r, float)
        if displacements is not None:
            if use_callable:
                d = np.array(displacements(aid, sp, r.copy()), float)
            else:
                d = np.array(displacements.get(aid, (0.0,0.0,0.0)), float)
            r = r + float(amp)*d
        lines.append(f" {sp:<3s} {r[0]:.12f} {r[1]:.12f} {r[2]:.12f}")
    return "\n".join(lines)

Use below for quantum ESPRESSO input files in equilibrium.

In [7]:
pos_id = auto_id_positions(positions_ang)
pos_id

[('Mn1', 'Mn1', [0.0, 0.0, 0.0]),
 ('Mn2', 'Mn2', [0.0, 0.0, 40.92600002659999]),
 ('Bi1', 'Bi', [0.0, 0.0, 17.389456335912]),
 ('Bi2', 'Bi', [0.0, 0.0, 58.315456362511995]),
 ('Bi3', 'Bi', [0.0, 0.0, 23.53654095084]),
 ('Bi4', 'Bi', [0.0, 0.0, 64.46254097744]),
 ('Te1', 'Te', [0.0, 0.0, 5.457892289616]),
 ('Te2', 'Te', [0.0, 0.0, 46.383889576368]),
 ('Te3', 'Te', [0.0, 0.0, 35.468107736984]),
 ('Te4', 'Te', [0.0, 0.0, 76.394107763584]),
 ('Te5', 'Te', [0.0, 0.0, 12.03470014456]),
 ('Te6', 'Te', [0.0, 0.0, 52.960697431312]),
 ('Te7', 'Te', [0.0, 0.0, 28.89129988204]),
 ('Te8', 'Te', [0.0, 0.0, 69.817297168792])]

In [8]:
print(make_qe_pos_block(pos_id))

ATOMIC_POSITIONS angstrom
 Mn1 0.000000000000 0.000000000000 0.000000000000
 Mn2 0.000000000000 0.000000000000 40.926000026600
 Bi  0.000000000000 0.000000000000 17.389456335912
 Bi  0.000000000000 0.000000000000 58.315456362512
 Bi  0.000000000000 0.000000000000 23.536540950840
 Bi  0.000000000000 0.000000000000 64.462540977440
 Te  0.000000000000 0.000000000000 5.457892289616
 Te  0.000000000000 0.000000000000 46.383889576368
 Te  0.000000000000 0.000000000000 35.468107736984
 Te  0.000000000000 0.000000000000 76.394107763584
 Te  0.000000000000 0.000000000000 12.034700144560
 Te  0.000000000000 0.000000000000 52.960697431312
 Te  0.000000000000 0.000000000000 28.891299882040
 Te  0.000000000000 0.000000000000 69.817297168792


# Phonon Displacements

SMODES prints out the irrep, degeneracy, and number of modes, followed by the superlattice vectors, equilibrium atomic positions, and phonon displacements. Each symmetry mode separated by a line of dashes is a separate mode.

Below is the T1+ irrep with 3 modes. This is inversion even in the non-magnetic cell.

## T1+ Irrep

```text
Irrep T1+            
Degeneracy:  1
Total number of modes:   3
Vectors defining superlattice
  -2.16680  -1.25100  27.28400
   2.16680  -1.25100  27.28400
   0.00000   2.50201  27.28400
atom  type   position
  1   Mn     0.00000   0.00000   0.00000
  2   Mn     0.00000   0.00000  40.92600
  3   Bi     0.00000   0.00000  17.38946
  4   Bi     0.00000   0.00000  58.31546
  5   Bi     0.00000   0.00000  64.46254
  6   Bi     0.00000   0.00000  23.53654
  7   Te1    0.00000   0.00000   5.45789
  8   Te1    0.00000   0.00000  46.38389
  9   Te1    0.00000   0.00000  76.39411
 10   Te1    0.00000   0.00000  35.46811
 11   Te2    0.00000   0.00000  12.03470
 12   Te2    0.00000   0.00000  52.96070
 13   Te2    0.00000   0.00000  69.81730
 14   Te2    0.00000   0.00000  28.89130
Symmetry modes:
atom  type   displacement
------------------------------------------
  3   Bi     0.00000   0.00000   1.00000
  4   Bi    -0.00000  -0.00000  -1.00000
  5   Bi    -0.00000  -0.00000  -1.00000
  6   Bi     0.00000   0.00000   1.00000
------------------------------------------
  7   Te1    0.00000   0.00000   1.00000
  8   Te1   -0.00000  -0.00000  -1.00000
  9   Te1   -0.00000  -0.00000  -1.00000
 10   Te1    0.00000   0.00000   1.00000
------------------------------------------
 11   Te2    0.00000   0.00000   1.00000
 12   Te2   -0.00000  -0.00000  -1.00000
 13   Te2   -0.00000  -0.00000  -1.00000
 14   Te2    0.00000   0.00000   1.00000
 ```

```text
1 Mn -> Mn1
2 Mn -> Mn2
3 Bi -> Bi1
4 Bi -> Bi2
5 Bi -> Bi4
6 Bi -> Bi3
7 Te1 -> Te1
8 Te1 -> Te2
9 Te1 -> Te4
10 Te1 -> Te3
11 Te2 -> Te5
12 Te2 -> Te6
13 Te2 -> Te8
14 Te2 -> Te7
```

We will use these displacements to create displaced POSITIONS files for Quantum ESPRESSO calculations.

In [9]:
amps = [0, 0.01]

In [10]:
displacements = {
    "mode 1": 
    {
        "Bi1": [0, 0,  1],
        "Bi2": [0, 0, -1],
        "Bi3": [0, 0,  1],
        "Bi4": [0, 0, -1],
    },
    "mode 2": {
        "Te1": [0, 0,  1],
        "Te2": [0, 0, -1],
        "Te3": [0, 0,  1],
        "Te4": [0, 0, -1],
    },
    "mode 3":
    {
        "Te5": [0, 0,  1],
        "Te6": [0, 0, -1],
        "Te7": [0, 0,  1],
        "Te8": [0, 0, -1],
    }
}

### Mode 1

In [11]:
print(displacements["mode 1"])

{'Bi1': [0, 0, 1], 'Bi2': [0, 0, -1], 'Bi3': [0, 0, 1], 'Bi4': [0, 0, -1]}


In [12]:
for amp in amps:
    print(f"Amplitude = {amp} angstrom")
    print("Quantum ESPRESSO POSITIONS with displacements:\n")
    print(make_qe_pos_block(pos_id, displacements["mode 1"], amp))
    print()
    print("----------------------------------------------")

Amplitude = 0 angstrom
Quantum ESPRESSO POSITIONS with displacements:

ATOMIC_POSITIONS angstrom
 Mn1 0.000000000000 0.000000000000 0.000000000000
 Mn2 0.000000000000 0.000000000000 40.926000026600
 Bi  0.000000000000 0.000000000000 17.389456335912
 Bi  0.000000000000 0.000000000000 58.315456362512
 Bi  0.000000000000 0.000000000000 23.536540950840
 Bi  0.000000000000 0.000000000000 64.462540977440
 Te  0.000000000000 0.000000000000 5.457892289616
 Te  0.000000000000 0.000000000000 46.383889576368
 Te  0.000000000000 0.000000000000 35.468107736984
 Te  0.000000000000 0.000000000000 76.394107763584
 Te  0.000000000000 0.000000000000 12.034700144560
 Te  0.000000000000 0.000000000000 52.960697431312
 Te  0.000000000000 0.000000000000 28.891299882040
 Te  0.000000000000 0.000000000000 69.817297168792

----------------------------------------------
Amplitude = 0.01 angstrom
Quantum ESPRESSO POSITIONS with displacements:

ATOMIC_POSITIONS angstrom
 Mn1 0.000000000000 0.000000000000 0.000000

### Mode 2

In [13]:
print(displacements["mode 2"])

{'Te1': [0, 0, 1], 'Te2': [0, 0, -1], 'Te3': [0, 0, 1], 'Te4': [0, 0, -1]}


In [14]:
for amp in amps:
    print(f"Amplitude = {amp} angstrom")
    print("Quantum ESPRESSO POSITIONS with displacements:\n")
    print(make_qe_pos_block(pos_id, displacements["mode 2"], amp))
    print()
    print("----------------------------------------------")

Amplitude = 0 angstrom
Quantum ESPRESSO POSITIONS with displacements:

ATOMIC_POSITIONS angstrom
 Mn1 0.000000000000 0.000000000000 0.000000000000
 Mn2 0.000000000000 0.000000000000 40.926000026600
 Bi  0.000000000000 0.000000000000 17.389456335912
 Bi  0.000000000000 0.000000000000 58.315456362512
 Bi  0.000000000000 0.000000000000 23.536540950840
 Bi  0.000000000000 0.000000000000 64.462540977440
 Te  0.000000000000 0.000000000000 5.457892289616
 Te  0.000000000000 0.000000000000 46.383889576368
 Te  0.000000000000 0.000000000000 35.468107736984
 Te  0.000000000000 0.000000000000 76.394107763584
 Te  0.000000000000 0.000000000000 12.034700144560
 Te  0.000000000000 0.000000000000 52.960697431312
 Te  0.000000000000 0.000000000000 28.891299882040
 Te  0.000000000000 0.000000000000 69.817297168792

----------------------------------------------
Amplitude = 0.01 angstrom
Quantum ESPRESSO POSITIONS with displacements:

ATOMIC_POSITIONS angstrom
 Mn1 0.000000000000 0.000000000000 0.000000

### Mode 3

In [15]:
print(displacements["mode 3"])

{'Te5': [0, 0, 1], 'Te6': [0, 0, -1], 'Te7': [0, 0, 1], 'Te8': [0, 0, -1]}


In [16]:
for amp in amps:
    print(f"Amplitude = {amp} angstrom")
    print("Quantum ESPRESSO POSITIONS with displacements:\n")
    print(make_qe_pos_block(pos_id, displacements["mode 3"], amp))
    print()
    print("----------------------------------------------")

Amplitude = 0 angstrom
Quantum ESPRESSO POSITIONS with displacements:

ATOMIC_POSITIONS angstrom
 Mn1 0.000000000000 0.000000000000 0.000000000000
 Mn2 0.000000000000 0.000000000000 40.926000026600
 Bi  0.000000000000 0.000000000000 17.389456335912
 Bi  0.000000000000 0.000000000000 58.315456362512
 Bi  0.000000000000 0.000000000000 23.536540950840
 Bi  0.000000000000 0.000000000000 64.462540977440
 Te  0.000000000000 0.000000000000 5.457892289616
 Te  0.000000000000 0.000000000000 46.383889576368
 Te  0.000000000000 0.000000000000 35.468107736984
 Te  0.000000000000 0.000000000000 76.394107763584
 Te  0.000000000000 0.000000000000 12.034700144560
 Te  0.000000000000 0.000000000000 52.960697431312
 Te  0.000000000000 0.000000000000 28.891299882040
 Te  0.000000000000 0.000000000000 69.817297168792

----------------------------------------------
Amplitude = 0.01 angstrom
Quantum ESPRESSO POSITIONS with displacements:

ATOMIC_POSITIONS angstrom
 Mn1 0.000000000000 0.000000000000 0.000000

## T2- Irrep

```text
***********************************************
Irrep T2-            
Degeneracy:  1
Total number of modes:   4
Vectors defining superlattice
  -2.16680  -1.25100  27.28400
   2.16680  -1.25100  27.28400
   0.00000   2.50201  27.28400
atom  type   position
  1   Mn     0.00000   0.00000   0.00000
  2   Mn     0.00000   0.00000  40.92600
  3   Bi     0.00000   0.00000  17.38946
  4   Bi     0.00000   0.00000  58.31546
  5   Bi     0.00000   0.00000  64.46254
  6   Bi     0.00000   0.00000  23.53654
  7   Te1    0.00000   0.00000   5.45953
  8   Te1    0.00000   0.00000  46.38553
  9   Te1    0.00000   0.00000  76.39247
 10   Te1    0.00000   0.00000  35.46647
 11   Te2    0.00000   0.00000  12.03634
 12   Te2    0.00000   0.00000  52.96234
 13   Te2    0.00000   0.00000  69.81566
 14   Te2    0.00000   0.00000  28.88966
Symmetry modes:
atom  type   displacement
------------------------------------------
  1   Mn     0.00000   0.00000   1.00000
  2   Mn    -0.00000  -0.00000  -1.00000
------------------------------------------
  3   Bi     0.00000   0.00000   1.00000
  4   Bi    -0.00000  -0.00000  -1.00000
  5   Bi     0.00000   0.00000   1.00000
  6   Bi    -0.00000  -0.00000  -1.00000
------------------------------------------
  7   Te1    0.00000   0.00000   1.00000
  8   Te1   -0.00000  -0.00000  -1.00000
  9   Te1    0.00000   0.00000   1.00000
 10   Te1   -0.00000  -0.00000  -1.00000
------------------------------------------
 11   Te2    0.00000   0.00000   1.00000
 12   Te2   -0.00000  -0.00000  -1.00000
 13   Te2    0.00000   0.00000   1.00000
 14   Te2   -0.00000  -0.00000  -1.00000
***********************************************
 ```

We will use these displacements to create displaced POSITIONS files for Quantum ESPRESSO calculations.

In [17]:
amps = [0, 0.01, 0.02, 0.03, 0.04]

In [18]:
displacements = {
    "mode 1": 
    {
        "Mn1": [0, 0,  1],
        "Mn2": [0, 0, -1]
    },
    "mode 2": {
        "Bi1": [0, 0,  1],
        "Bi2": [0, 0, -1],
        "Bi4": [0, 0,  1],
        "Bi3": [0, 0, -1],
    },
    "mode 3": {
        "Te1": [0, 0,  1],
        "Te2": [0, 0, -1],
        "Te4": [0, 0,  1],
        "Te3": [0, 0, -1],
    },
    "mode 4":
    {
        "Te5": [0, 0,  1],
        "Te6": [0, 0, -1],
        "Te8": [0, 0,  1],
        "Te7": [0, 0, -1],
    }
}

### Mode 1
```text
  1   Mn     0.00000   0.00000   1.00000
  2   Mn    -0.00000  -0.00000  -1.00000
```

We will generate the Quantum ESPRESSO ATOMIC_POSITIONS block for this mode. To generate the displaced atomic positions for Quantum ESPRESSO input files, we take the equilibrium atomic positions and add the displacements multiplied by an amplitude factor `amp` (in Angstroms).

In [19]:
for amp in amps:
    print(f"Amplitude = {amp} angstrom")
    print("Quantum ESPRESSO POSITIONS with displacements:\n")
    print(make_qe_pos_block(pos_id, displacements["mode 1"], amp))
    print()
    print("----------------------------------------------")

Amplitude = 0 angstrom
Quantum ESPRESSO POSITIONS with displacements:

ATOMIC_POSITIONS angstrom
 Mn1 0.000000000000 0.000000000000 0.000000000000
 Mn2 0.000000000000 0.000000000000 40.926000026600
 Bi  0.000000000000 0.000000000000 17.389456335912
 Bi  0.000000000000 0.000000000000 58.315456362512
 Bi  0.000000000000 0.000000000000 23.536540950840
 Bi  0.000000000000 0.000000000000 64.462540977440
 Te  0.000000000000 0.000000000000 5.457892289616
 Te  0.000000000000 0.000000000000 46.383889576368
 Te  0.000000000000 0.000000000000 35.468107736984
 Te  0.000000000000 0.000000000000 76.394107763584
 Te  0.000000000000 0.000000000000 12.034700144560
 Te  0.000000000000 0.000000000000 52.960697431312
 Te  0.000000000000 0.000000000000 28.891299882040
 Te  0.000000000000 0.000000000000 69.817297168792

----------------------------------------------
Amplitude = 0.01 angstrom
Quantum ESPRESSO POSITIONS with displacements:

ATOMIC_POSITIONS angstrom
 Mn1 0.000000000000 0.000000000000 0.010000

### Mode 2
```text
  3   Bi     0.00000   0.00000   1.00000
  4   Bi    -0.00000  -0.00000  -1.00000
  5   Bi     0.00000   0.00000   1.00000
  6   Bi    -0.00000  -0.00000  -1.00000
```

In [20]:
print(displacements["mode 2"])

{'Bi1': [0, 0, 1], 'Bi2': [0, 0, -1], 'Bi4': [0, 0, 1], 'Bi3': [0, 0, -1]}


In [21]:
for amp in amps:
    print(f"Amplitude = {amp} angstrom")
    print("Quantum ESPRESSO POSITIONS with displacements:\n")
    print(make_qe_pos_block(pos_id, displacements["mode 2"], amp))
    print()
    print("----------------------------------------------")

Amplitude = 0 angstrom
Quantum ESPRESSO POSITIONS with displacements:

ATOMIC_POSITIONS angstrom
 Mn1 0.000000000000 0.000000000000 0.000000000000
 Mn2 0.000000000000 0.000000000000 40.926000026600
 Bi  0.000000000000 0.000000000000 17.389456335912
 Bi  0.000000000000 0.000000000000 58.315456362512
 Bi  0.000000000000 0.000000000000 23.536540950840
 Bi  0.000000000000 0.000000000000 64.462540977440
 Te  0.000000000000 0.000000000000 5.457892289616
 Te  0.000000000000 0.000000000000 46.383889576368
 Te  0.000000000000 0.000000000000 35.468107736984
 Te  0.000000000000 0.000000000000 76.394107763584
 Te  0.000000000000 0.000000000000 12.034700144560
 Te  0.000000000000 0.000000000000 52.960697431312
 Te  0.000000000000 0.000000000000 28.891299882040
 Te  0.000000000000 0.000000000000 69.817297168792

----------------------------------------------
Amplitude = 0.01 angstrom
Quantum ESPRESSO POSITIONS with displacements:

ATOMIC_POSITIONS angstrom
 Mn1 0.000000000000 0.000000000000 0.000000

### Mode 3

```text
7   Te1    0.00000   0.00000   1.00000
8   Te1   -0.00000  -0.00000  -1.00000
9   Te1    0.00000   0.00000   1.00000
10   Te1   -0.00000  -0.00000  -1.00000
```

In [22]:
print(displacements["mode 3"])

{'Te1': [0, 0, 1], 'Te2': [0, 0, -1], 'Te4': [0, 0, 1], 'Te3': [0, 0, -1]}


In [23]:
for amp in amps:
    print(f"Amplitude = {amp} angstrom")
    print("Quantum ESPRESSO POSITIONS with displacements:\n")
    print(make_qe_pos_block(pos_id, displacements["mode 3"], amp))
    print()
    print("----------------------------------------------")

Amplitude = 0 angstrom
Quantum ESPRESSO POSITIONS with displacements:

ATOMIC_POSITIONS angstrom
 Mn1 0.000000000000 0.000000000000 0.000000000000
 Mn2 0.000000000000 0.000000000000 40.926000026600
 Bi  0.000000000000 0.000000000000 17.389456335912
 Bi  0.000000000000 0.000000000000 58.315456362512
 Bi  0.000000000000 0.000000000000 23.536540950840
 Bi  0.000000000000 0.000000000000 64.462540977440
 Te  0.000000000000 0.000000000000 5.457892289616
 Te  0.000000000000 0.000000000000 46.383889576368
 Te  0.000000000000 0.000000000000 35.468107736984
 Te  0.000000000000 0.000000000000 76.394107763584
 Te  0.000000000000 0.000000000000 12.034700144560
 Te  0.000000000000 0.000000000000 52.960697431312
 Te  0.000000000000 0.000000000000 28.891299882040
 Te  0.000000000000 0.000000000000 69.817297168792

----------------------------------------------
Amplitude = 0.01 angstrom
Quantum ESPRESSO POSITIONS with displacements:

ATOMIC_POSITIONS angstrom
 Mn1 0.000000000000 0.000000000000 0.000000

### Mode 4
```text
 11   Te2    0.00000   0.00000   1.00000
 12   Te2   -0.00000  -0.00000  -1.00000
 13   Te2    0.00000   0.00000   1.00000
 14   Te2   -0.00000  -0.00000  -1.00000
 ```

In [24]:
print(displacements["mode 4"])

{'Te5': [0, 0, 1], 'Te6': [0, 0, -1], 'Te8': [0, 0, 1], 'Te7': [0, 0, -1]}


In [25]:
for amp in amps:
    print(f"Amplitude = {amp} angstrom")
    print("Quantum ESPRESSO POSITIONS with displacements:\n")
    print(make_qe_pos_block(pos_id, displacements["mode 4"], amp))
    print()
    print("----------------------------------------------")

Amplitude = 0 angstrom
Quantum ESPRESSO POSITIONS with displacements:

ATOMIC_POSITIONS angstrom
 Mn1 0.000000000000 0.000000000000 0.000000000000
 Mn2 0.000000000000 0.000000000000 40.926000026600
 Bi  0.000000000000 0.000000000000 17.389456335912
 Bi  0.000000000000 0.000000000000 58.315456362512
 Bi  0.000000000000 0.000000000000 23.536540950840
 Bi  0.000000000000 0.000000000000 64.462540977440
 Te  0.000000000000 0.000000000000 5.457892289616
 Te  0.000000000000 0.000000000000 46.383889576368
 Te  0.000000000000 0.000000000000 35.468107736984
 Te  0.000000000000 0.000000000000 76.394107763584
 Te  0.000000000000 0.000000000000 12.034700144560
 Te  0.000000000000 0.000000000000 52.960697431312
 Te  0.000000000000 0.000000000000 28.891299882040
 Te  0.000000000000 0.000000000000 69.817297168792

----------------------------------------------
Amplitude = 0.01 angstrom
Quantum ESPRESSO POSITIONS with displacements:

ATOMIC_POSITIONS angstrom
 Mn1 0.000000000000 0.000000000000 0.000000